# 02 — Graph Construction · Phase 3

**Purpose:** Load the built graph, verify every assertion, visualise the node distribution, edge structure, feature correlations, and geographic split boundaries.

**Pre-condition:** Run `python scripts/phase3_build_graph.py` first. This notebook is for verification and exploration only — it does not rebuild the graph.

**What you confirm here before Phase 4:**
- Graph has the correct number of nodes, features, and edges
- val_mask is not zero (previous project bug)
- No geographic overlap between train/val/test splits
- Target variable y is quantile-transformed (mean≈0, std≈1)
- Feature matrix has no NaN or Inf values
- Node distribution across Greece looks spatially correct

---

In [1]:
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

import torch
from torch_geometric.data import Data

# Add src/ to path
PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from wildfire_gnn.utils import load_yaml_config, set_seed

config  = load_yaml_config(PROJECT_ROOT / 'configs' / 'gnn_config.yaml')
set_seed(config['training']['seed'])

p           = config['paths']
GRAPH_PATH  = PROJECT_ROOT / p['graph_data']
SPLIT_PATH  = PROJECT_ROOT / p['spatial_split_path']
FEAT_PATH   = PROJECT_ROOT / p['feature_names']
FIG_DIR     = PROJECT_ROOT / p['figures_dir']
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Graph path   : {GRAPH_PATH}')
print(f'Graph exists : {GRAPH_PATH.exists()}')

Project root : d:\wildfire\spatiotemporal_wildfire_gnn
Graph path   : d:\wildfire\spatiotemporal_wildfire_gnn\data\processed\graph_data_enriched.pt
Graph exists : True


# Load graph and build summary

In [2]:
if not GRAPH_PATH.exists():
    print('ERROR: graph_data_enriched.pt not found.')
    print('Run: python scripts/phase3_build_graph.py')
    raise FileNotFoundError(str(GRAPH_PATH))

graph = torch.load(GRAPH_PATH, map_location='cpu', weights_only=False)

print('Graph loaded successfully')
print()
print(f'  num_nodes          : {graph.num_nodes:,}')
print(f'  num_node_features  : {graph.num_node_features}')
print(f'  num_edges          : {graph.num_edges:,}')
print(f'  avg edges per node : {graph.num_edges / graph.num_nodes:.1f}')
print()
print(f'  x shape            : {tuple(graph.x.shape)}')
print(f'  y shape            : {tuple(graph.y.shape)}')
print(f'  pos shape          : {tuple(graph.pos.shape)}')
print(f'  edge_index shape   : {tuple(graph.edge_index.shape)}')
print()
print(f'  train_mask.sum()   : {graph.train_mask.sum():,}')
print(f'  val_mask.sum()     : {graph.val_mask.sum():,}')
print(f'  test_mask.sum()    : {graph.test_mask.sum():,}')
print()
print(f'  y mean             : {float(graph.y.mean()):.4f}  (should be near 0)')
print(f'  y std              : {float(graph.y.std()):.4f}   (should be near 1)')
print(f'  y_raw min          : {float(graph.y_raw.min()):.6f}')
print(f'  y_raw max          : {float(graph.y_raw.max()):.4f}')

Graph loaded successfully

  num_nodes          : 327,405
  num_node_features  : 61
  num_edges          : 2,511,084
  avg edges per node : 7.7

  x shape            : (327405, 61)
  y shape            : (327405, 1)
  pos shape          : (327405, 2)
  edge_index shape   : (2, 2511084)

  train_mask.sum()   : 68,557
  val_mask.sum()     : 11,352
  test_mask.sum()    : 247,496

  y mean             : 0.0077  (should be near 0)
  y std              : 0.9930   (should be near 1)
  y_raw min          : 0.000011
  y_raw max          : 0.2505


# Critical assertion

In [3]:
print('Running Phase 3 completion assertions...')
print('=' * 55)

failures = []

def check(condition, message, fix=''):
    sym = '✓' if condition else '✗'
    print(f'  {sym}  {message}')
    if not condition:
        failures.append((message, fix))

# Node count
check(graph.num_nodes > 100_000,
      f'num_nodes={graph.num_nodes:,} > 100k',
      'Re-run phase3_build_graph.py with --stride 6')

check(graph.num_nodes < 600_000,
      f'num_nodes={graph.num_nodes:,} < 600k (memory feasible)',
      'Use --stride 6 or higher')

# Feature count
check(graph.num_node_features >= 58,
      f'num_features={graph.num_node_features} >= 58 (DEM included; fuel categories may add extra features)',
      'Check DEM features and feature_engineering.py output')

# Edge count
check(graph.num_edges > graph.num_nodes * 2,
      f'num_edges={graph.num_edges:,} > 2×nodes (graph is connected)',
      'Check build_pixel_grid_edges() in graph_builder.py')

# Target transformation
y_mean = float(graph.y.mean())
y_std  = float(graph.y.std())
check(abs(y_mean) < 0.5,
      f'y.mean()={y_mean:.4f} near 0 (QuantileTransformer applied)',
      'Transform was not applied or was applied twice')

check(0.5 < y_std < 2.0,
      f'y.std()={y_std:.4f} near 1 (QuantileTransformer applied)',
      'Check TargetTransformer.transform() in target_engineering.py')

# No NaN in features
has_nan = torch.isnan(graph.x).any()
check(not has_nan,
      'Feature matrix x has no NaN values',
      'Check nan_to_num() calls in feature_engineering.py')

# No Inf in features
has_inf = torch.isinf(graph.x).any()
check(not has_inf,
      'Feature matrix x has no Inf values',
      'Check gradient computation in add_spatial_gradients()')

# Split masks non-zero
check(graph.val_mask.sum() > 0,
      f'val_mask.sum()={graph.val_mask.sum():,} > 0 (not zero placeholder)',
      'CRITICAL: val rows range in config produces no nodes — check split.val_rows')

check(graph.test_mask.sum() > 0,
      f'test_mask.sum()={graph.test_mask.sum():,} > 0',
      'Check split.test_rows range in gnn_config.yaml')

# No geographic overlap
tv_overlap = int((graph.train_mask & graph.val_mask).sum())
tt_overlap = int((graph.train_mask & graph.test_mask).sum())
check(tv_overlap == 0,
      f'Train/Val overlap = {tv_overlap} (must be 0)',
      'CRITICAL geographic leakage — check build_geographic_split()')

check(tt_overlap == 0,
      f'Train/Test overlap = {tt_overlap} (must be 0)',
      'CRITICAL geographic leakage — check build_geographic_split()')

# All nodes covered
covered = int(graph.train_mask.sum() + graph.val_mask.sum() + graph.test_mask.sum())
check(covered == graph.num_nodes,
      f'All {graph.num_nodes:,} nodes covered by exactly one split',
      'Check test_rows upper bound in gnn_config.yaml')

# edge_index valid range
check(int(graph.edge_index.max()) < graph.num_nodes,
      f'edge_index max={int(graph.edge_index.max())} < num_nodes={graph.num_nodes}',
      'Edge index contains out-of-range node index')

# dtype checks
check(graph.x.dtype == torch.float32,
      f'x.dtype = {graph.x.dtype} (float32)',
      'Cast x to float32 in graph assembly')

check(graph.edge_index.dtype == torch.long,
      f'edge_index.dtype = {graph.edge_index.dtype} (torch.long/int64)',
      'Cast edge_index to torch.long')

print('=' * 55)
if failures:
    print(f'\n  FAILED: {len(failures)} assertion(s)')
    for msg, fix in failures:
        print(f'  ✗  {msg}')
        if fix:
            print(f'     Fix: {fix}')
    print('\n  Do NOT proceed to Phase 4 until all assertions pass.')
else:
    print(f'\n  ALL {15} ASSERTIONS PASSED — ready for Phase 4')

Running Phase 3 completion assertions...
  ✓  num_nodes=327,405 > 100k
  ✓  num_nodes=327,405 < 600k (memory feasible)
  ✓  num_features=61 >= 58 (DEM included; fuel categories may add extra features)
  ✓  num_edges=2,511,084 > 2×nodes (graph is connected)
  ✓  y.mean()=0.0077 near 0 (QuantileTransformer applied)
  ✓  y.std()=0.9930 near 1 (QuantileTransformer applied)
  ✓  Feature matrix x has no NaN values
  ✓  Feature matrix x has no Inf values
  ✓  val_mask.sum()=11,352 > 0 (not zero placeholder)
  ✓  test_mask.sum()=247,496 > 0
  ✓  Train/Val overlap = 0 (must be 0)
  ✓  Train/Test overlap = 0 (must be 0)
  ✓  All 327,405 nodes covered by exactly one split
  ✓  edge_index max=327404 < num_nodes=327405
  ✓  x.dtype = torch.float32 (float32)
  ✓  edge_index.dtype = torch.int64 (torch.long/int64)

  ALL 15 ASSERTIONS PASSED — ready for Phase 4


# Feature names and group breakdown

In [4]:
if FEAT_PATH.exists():
    with open(FEAT_PATH) as f:
        feature_names = json.load(f)
    print(f'Total features: {len(feature_names)}')
    print()

    groups = [
        ('Base rasters',      [n for n in feature_names if n in ['CFL','FSP_Index','Ignition_Prob','Struct_Exp_Index']]),
        ('DEM terrain',       [n for n in feature_names if n.startswith('dem_')]),
        ('Fuel one-hot',      [n for n in feature_names if n.startswith('fuel_')]),
        ('Interactions',      [n for n in feature_names if n.startswith('interact_')]),
        ('Multi-scale stats', [n for n in feature_names if '_mean_' in n or '_std_' in n]),
        ('Spatial gradients', [n for n in feature_names if '_grad_' in n]),
        ('Node degree',       [n for n in feature_names if n == 'node_degree']),
    ]

    print(f'  {"Group":<22} {"Count":<8} Features')
    print(f'  {"-"*70}')
    for gname, gfeats in groups:
        feat_str = ', '.join(gfeats[:4])
        if len(gfeats) > 4:
            feat_str += f' ... (+{len(gfeats)-4} more)'
        print(f'  {gname:<22} {len(gfeats):<8} {feat_str}')
    total = sum(len(g) for _, g in groups)
    print(f'  {"-"*70}')
    print(f'  {"TOTAL":<22} {total}')
else:
    print('feature_names.json not found — run phase3_build_graph.py first')

Total features: 61

  Group                  Count    Features
  ----------------------------------------------------------------------
  Base rasters           4        CFL, FSP_Index, Ignition_Prob, Struct_Exp_Index
  DEM terrain            5        dem_elevation_m, dem_slope_deg, dem_aspect_sin, dem_aspect_cos ... (+1 more)
  Fuel one-hot           24       fuel_91, fuel_93, fuel_98, fuel_99 ... (+20 more)
  Interactions           3        interact_CFL_Ignition, interact_FSP_CFL, interact_Ignition_FSP
  Multi-scale stats      18       CFL_mean_3x3, CFL_std_3x3, CFL_mean_7x7, CFL_std_7x7 ... (+14 more)
  Spatial gradients      6        CFL_grad_x, CFL_grad_y, FSP_Index_grad_x, FSP_Index_grad_y ... (+2 more)
  Node degree            1        node_degree
  ----------------------------------------------------------------------
  TOTAL                  61


# Split Statistic table

In [5]:
splits = {
    'train': graph.train_mask,
    'val':   graph.val_mask,
    'test':  graph.test_mask,
}

print(f'Split statistics  (N = {graph.num_nodes:,} total nodes)')
print('=' * 70)
print(f'  {"Split":<8} {"Nodes":>9} {"Pct":>7}  {"BP mean":>9} {"BP std":>9} '
      f'{"Row min":>9} {"Row max":>9}')
print(f'  {"-"*68}')

for name, mask in splits.items():
    idx      = mask.nonzero(as_tuple=True)[0]
    n        = int(mask.sum())
    pct      = 100 * n / graph.num_nodes
    bp_vals  = graph.y_raw[idx].numpy().ravel()
    row_vals = graph.pos[idx, 0].numpy()
    print(f'  {name:<8} {n:>9,} {pct:>6.1f}%  '
          f'{bp_vals.mean():>9.5f} {bp_vals.std():>9.5f} '
          f'{int(row_vals.min()):>9} {int(row_vals.max()):>9}')

print('=' * 70)
print()
print('Geographic disjointness verification:')
print(f'  train rows: {int(graph.pos[graph.train_mask][:,0].min())} — {int(graph.pos[graph.train_mask][:,0].max())}')
print(f'  val   rows: {int(graph.pos[graph.val_mask][:,0].min())} — {int(graph.pos[graph.val_mask][:,0].max())}')
print(f'  test  rows: {int(graph.pos[graph.test_mask][:,0].min())} — {int(graph.pos[graph.test_mask][:,0].max())}')
print()

# Confirm the row ranges are disjoint
train_max = int(graph.pos[graph.train_mask][:,0].max())
val_min   = int(graph.pos[graph.val_mask][:,0].min())
val_max   = int(graph.pos[graph.val_mask][:,0].max())
test_min  = int(graph.pos[graph.test_mask][:,0].min())

assert train_max < val_min, f'Train bleeds into Val: train_max={train_max}, val_min={val_min}'
assert val_max   < test_min, f'Val bleeds into Test: val_max={val_max}, test_min={test_min}'
print('✓  Row ranges are geographically disjoint — no leakage')

Split statistics  (N = 327,405 total nodes)
  Split        Nodes     Pct    BP mean    BP std   Row min   Row max
  --------------------------------------------------------------------
  train       68,557   20.9%    0.01317   0.01750         6      1326
  val         11,352    3.5%    0.02054   0.03011      1332      1512
  test       247,496   75.6%    0.02765   0.03557      1518      7590

Geographic disjointness verification:
  train rows: 6 — 1326
  val   rows: 1332 — 1512
  test  rows: 1518 — 7590

✓  Row ranges are geographically disjoint — no leakage


# Feature statistics per group

In [6]:
if not FEAT_PATH.exists():
    print('feature_names.json not found')
else:
    with open(FEAT_PATH) as f:
        feature_names = json.load(f)

    X = graph.x.numpy()

    print(f'Feature matrix: {X.shape}')
    print(f'NaN count     : {np.isnan(X).sum()}')
    print(f'Inf count     : {np.isinf(X).sum()}')
    print()

    print(f'  {"Feature":<30} {"mean":>10} {"std":>10} {"min":>10} {"max":>10}')
    print(f'  {"-"*72}')

    groups_order = [
        ('BASE', ['CFL','FSP_Index','Ignition_Prob','Struct_Exp_Index']),
        ('DEM',  [n for n in feature_names if n.startswith('dem_')]),
        ('FUEL (first 3)', [n for n in feature_names if n.startswith('fuel_')][:3]),
        ('INTERACT', [n for n in feature_names if n.startswith('interact_')]),
        ('MULTISCALE (sample)', [n for n in feature_names if '_mean_3x3' in n][:3]),
        ('GRADIENT (sample)',   [n for n in feature_names if '_grad_x' in n][:3]),
    ]

    for gname, gfeats in groups_order:
        if not gfeats:
            continue
        print(f'  ── {gname}')
        for fname in gfeats:
            if fname not in feature_names:
                continue
            idx = feature_names.index(fname)
            col = X[:, idx]
            print(f'  {fname:<30} {col.mean():>10.4f} {col.std():>10.4f} '
                  f'{col.min():>10.4f} {col.max():>10.4f}')
    print()

Feature matrix: (327405, 61)
NaN count     : 0
Inf count     : 0

  Feature                              mean        std        min        max
  ------------------------------------------------------------------------
  ── BASE
  CFL                                3.8133     3.4424     1.0000    25.0000
  FSP_Index                       1110.8486  1594.1091     1.5070 37708.6523
  Ignition_Prob                      0.2945     0.2206     0.0000     0.9924
  Struct_Exp_Index                 236.7563   323.1908     0.0000  6276.8574
  ── DEM
  dem_elevation_m                  519.2512   430.2966    -7.0000  2820.4666
  dem_slope_deg                     11.6343     8.6916     0.0000    76.8225
  dem_aspect_sin                    -0.0108     0.7101    -1.0000     1.0000
  dem_aspect_cos                     0.0456     0.7022    -1.0000     1.0000
  dem_twi                            1.9826     1.1160    -1.4519    10.9560
  ── FUEL (first 3)
  fuel_91                            0.0251     0.

# Memory footprint estimate for GPU training

In [7]:
N = graph.num_nodes
F = graph.num_node_features
E = graph.num_edges
H = config['model']['hidden_channels']
heads = config['model']['heads']

# Memory estimate (bytes)
x_bytes          = N * F * 4          # float32
edge_bytes       = E * 2 * 8          # int64 pairs
y_bytes          = N * 1 * 4          # float32
hidden_bytes     = N * H * 4          # one hidden layer
attention_bytes  = E * heads * 4      # attention coefficients

total_data   = x_bytes + edge_bytes + y_bytes
total_model  = hidden_bytes + attention_bytes
total_est    = total_data + total_model * config['model']['num_layers']

def mb(b): return b / 1024**2

print('GPU memory estimate for full-graph training:')
print(f'  Node feature matrix x  : {mb(x_bytes):.1f} MB')
print(f'  Edge index             : {mb(edge_bytes):.1f} MB')
print(f'  Target y               : {mb(y_bytes):.1f} MB')
print(f'  Hidden states (1 layer): {mb(hidden_bytes):.1f} MB')
print(f'  Attention coeffs (8h)  : {mb(attention_bytes):.1f} MB')
print(f'  ─────────────────────────────────────────')
print(f'  Estimated minimum VRAM : {mb(total_est):.1f} MB  ({mb(total_est)/1024:.1f} GB)')
print(f'  Add 2–3× for gradients + activations')
print(f'  Recommended VRAM       : {mb(total_est*2.5)/1024:.0f}–{mb(total_est*3)/1024:.0f} GB')
print()

if mb(total_est*2.5) > 16*1024:
    print('⚠  Estimated usage exceeds 16 GB — consider reducing --stride')
    print('   or enabling mini-batch with config training.batch_size=2048')
else:
    print('✓  Estimated usage fits within 16 GB VRAM — full-graph training feasible')

GPU memory estimate for full-graph training:
  Node feature matrix x  : 76.2 MB
  Edge index             : 38.3 MB
  Target y               : 1.2 MB
  Hidden states (1 layer): 319.7 MB
  Attention coeffs (8h)  : 76.6 MB
  ─────────────────────────────────────────
  Estimated minimum VRAM : 1701.2 MB  (1.7 GB)
  Add 2–3× for gradients + activations
  Recommended VRAM       : 4–5 GB

✓  Estimated usage fits within 16 GB VRAM — full-graph training feasible


# Phase 3 completion checklist

In [8]:
print('=' * 55)
print('  PHASE 3 COMPLETION CHECKLIST')
print('=' * 55)

items = [
    ('graph_data_enriched.pt saved',   GRAPH_PATH.exists()),
    ('splits_enriched.npz saved',      SPLIT_PATH.exists()),
    ('feature_names.json saved',       FEAT_PATH.exists()),
    ('num_nodes > 100k',               graph.num_nodes > 100_000),
    ('num_features 53 or 58',          graph.num_node_features in (53, 58)),
    ('y.mean() near 0',                abs(float(graph.y.mean())) < 0.5),
    ('y.std() near 1',                 0.5 < float(graph.y.std()) < 2.0),
    ('val_mask.sum() > 0',             int(graph.val_mask.sum()) > 0),
    ('No train/val overlap',           int((graph.train_mask & graph.val_mask).sum()) == 0),
    ('No train/test overlap',          int((graph.train_mask & graph.test_mask).sum()) == 0),
    ('All nodes covered',              int(graph.train_mask.sum()+graph.val_mask.sum()+graph.test_mask.sum()) == graph.num_nodes),
    ('No NaN in x',                    not torch.isnan(graph.x).any().item()),
    ('No Inf in x',                    not torch.isinf(graph.x).any().item()),
    ('edge_index dtype = int64',       graph.edge_index.dtype == torch.long),
    ('x dtype = float32',              graph.x.dtype == torch.float32),
]

all_ok = True
for label, ok in items:
    sym = '✓' if ok else '✗'
    print(f'  {sym}  {label}')
    all_ok = all_ok and ok

print('=' * 55)
if all_ok:
    print()
    print('  ALL CHECKS PASSED')
    print('  Ready for Phase 4: Baseline Models')
    print()
    print('  Load graph in Phase 4 notebook with:')
    print(f"  graph = torch.load('{GRAPH_PATH}', weights_only=False)")
    print(f'  assert graph.num_nodes > 100_000')
    print(f'  assert abs(float(graph.y.mean())) < 0.5')
else:
    print()
    print('  SOME CHECKS FAILED — do not proceed to Phase 4')
    print('  Fix failures above and re-run phase3_build_graph.py')

  PHASE 3 COMPLETION CHECKLIST
  ✓  graph_data_enriched.pt saved
  ✓  splits_enriched.npz saved
  ✓  feature_names.json saved
  ✓  num_nodes > 100k
  ✗  num_features 53 or 58
  ✓  y.mean() near 0
  ✓  y.std() near 1
  ✓  val_mask.sum() > 0
  ✓  No train/val overlap
  ✓  No train/test overlap
  ✓  All nodes covered
  ✓  No NaN in x
  ✓  No Inf in x
  ✓  edge_index dtype = int64
  ✓  x dtype = float32

  SOME CHECKS FAILED — do not proceed to Phase 4
  Fix failures above and re-run phase3_build_graph.py


graph_builder:

phase3_build_graph.py

=================================================================
  Phase 3 — Graph Construction
=================================================================

  [0] Pre-condition checks...
  ✓  All Phase 2 outputs found

  ✓  Transformer loaded: D:\wildfire\spatiotemporal_wildfire_gnn\data\features\target_transformer.pkl
  ✓  DEM found — terrain features will be included (58 total)
  [1/6] Spatial grid subsampling (stride=6)...
  Spatial grid subsampling (stride=6):
    Grid candidates : 1,596,420
    In valid mask   : 327,405
    Selected nodes  : 327,405

  [2/6] Feature engineering...

  Building feature matrix for 327,405 nodes...

  [1/7] Base rasters (4):
    ✓  CFL                      mean=3.8133  std=3.4424
    ✓  FSP_Index                mean=1110.8486  std=1594.1091
    ✓  Ignition_Prob            mean=0.2945  std=0.2206
    ✓  Struct_Exp_Index         mean=236.7563  std=323.1908

  [2/7] DEM terrain (5 features):
    ✓  dem_elevation_m          mean=519.2512  std=430.2966
    ✓  dem_slope_deg            mean=11.6343  std=8.6916
    ✓  dem_aspect_sin           mean=-0.0108  std=0.7101
    ✓  dem_aspect_cos           mean=0.0456  std=0.7022
    ✓  dem_twi                  mean=1.9826  std=1.1160

  [3/7] Fuel model one-hot (21 expected):
    ✓  Fuel_Models one-hot: 24 categories → 24 binary features

  [4/7] Interaction terms (3):
    ✓  Interaction terms: 3 features (CFL×Ign, FSP×CFL, Ign×FSP)

  [5/7] Multi-scale neighborhood statistics (18):
    ✓  Multi-scale stats: 3 rasters × 3 kernels × 2 stats = 18 features

  [6/7] Spatial gradients (6):
    ✓  Spatial gradients: 3 rasters × 2 = 6 features

  [7/7] Node degree (1):
    ✓  Node degree: mean=0.959  (interior=1.0, boundary<1.0)

  Feature matrix: (327405, 61)  (61 features)
  Feature breakdown:
    Base rasters         4
    DEM terrain          5
    Fuel one-hot         24
    Interactions         3
    Multi-scale          18
    Gradients            6
    Degree               1
    TOTAL                61

  [3/6] Loading and transforming target variable...
  ✓  Target transform validated: mean=0.0077, std=0.9930
  ✓  y_raw : min=0.00001  max=0.2505  mean=0.02437
  ✓  y_t   : min=-5.199   max=5.199   mean=0.0077

  [4/6] Building 8-connected pixel grid edges...

  Building 8-connected edges (stride=6)...
    Edges built     : 2,511,084
    Avg per node    : 7.7

  [5/6] Geographic block split...

  Geographic split (north→south by raster row):
    Train rows [0, 1327]:   68,557 nodes  (20.9%)
    Val   rows [1328, 1517]:     11,352 nodes  (3.5%)
    Test  rows [1518, 7590]:  247,496 nodes  (75.6%)
  ✓  No geographic overlap. All 327,405 nodes covered.

  [6/6] Assembling PyG Data object...

  Running final assertions...
  ✓  num_nodes          = 327,405
  ✓  num_node_features  = 61
  ✓  num_edges          = 2,511,084
  ✓  train_mask.sum()   = 68,557
  ✓  val_mask.sum()     = 11,352
  ✓  test_mask.sum()    = 247,496
  ✓  y mean = 0.0077  std = 0.9930
  ✓  No geographic overlap between splits

  ✓  Graph saved: D:\wildfire\spatiotemporal_wildfire_gnn\data\processed\graph_data_enriched.pt  (120.4 MB)
  ✓  Splits saved: D:\wildfire\spatiotemporal_wildfire_gnn\data\features\splits_enriched.npz
  ✓  Feature names: D:\wildfire\spatiotemporal_wildfire_gnn\data\features\feature_names.json

=================================================================
  Phase 3 Complete — 0.9 min
=================================================================
  Nodes      : 327,405  (stride=6)
  Features   : 61
  Edges      : 2,511,084
  Train/Val/Test: 68,557 / 11,352 / 247,496
  Graph file : graph_data_enriched.pt

  Load in notebook:
    graph = torch.load('D:\wildfire\spatiotemporal_wildfire_gnn\data\processed\graph_data_enriched.pt')
    assert graph.num_nodes > 200_000
    assert graph.num_node_features >= 53
